# 02 — Train the Wisdom Language Model

Trains a small character-level language model on the wisdom literature corpus.  
Swap `ARCH` in the config cell to compare `"transformer"`, `"lstm"`, and `"rnn"` with everything else held constant.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from tokenizer import load_vocab, ENCODED_PATH
from dataset import load_datasets
from model import build_model, count_parameters

## Configuration

Change `ARCH` to switch architectures. Everything else stays the same.

In [ ]:
# ── Switch architecture here ──────────────────────────────────────────────────
ARCH        = "transformer"   # "transformer" | "lstm" | "rnn"

# ── Shared hyperparameters ────────────────────────────────────────────────────
CONTEXT_LEN = 128
EMBED_DIM   = 64
N_LAYERS    = 2
DROPOUT     = 0.1

# ── RNN / LSTM only ───────────────────────────────────────────────────────────
HIDDEN_DIM  = 256

# ── Transformer only ──────────────────────────────────────────────────────────
N_HEADS     = 4    # embed_dim must be divisible by n_heads
FF_DIM      = 256

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE  = 64
N_EPOCHS    = 10
LR          = 1e-3
VAL_SPLIT   = 0.1

# ── Checkpointing ─────────────────────────────────────────────────────────────
RESUME      = True   # auto-resume from latest checkpoint if one exists

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Data

In [ ]:
char_to_idx, idx_to_char = load_vocab()
VOCAB_SIZE = len(char_to_idx)
print(f"Vocab size: {VOCAB_SIZE}")

train_ds, val_ds = load_datasets(context_len=CONTEXT_LEN, val_split=VAL_SPLIT)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

## Model

In [ ]:
model = build_model(
    arch        = ARCH,
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    n_heads     = N_HEADS,
    n_layers    = N_LAYERS,
    ff_dim      = FF_DIM,
    context_len = CONTEXT_LEN,
    dropout     = DROPOUT,
).to(DEVICE)

print(f"Architecture : {ARCH}")
print(f"Parameters   : {count_parameters(model):,}")

## Training Loop

In [ ]:
import glob

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

LOG_EVERY = 100   # print a progress line every N batches

# ── Resume from latest checkpoint if available ────────────────────────────────
start_epoch  = 1
train_losses = []
val_losses   = []

if RESUME:
    ckpts = sorted(glob.glob(f"model_{ARCH}_ckpt_*.pt"))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch  = ckpt["epoch"] + 1
        train_losses = ckpt["train_losses"]
        val_losses   = ckpt["val_losses"]
        print(f"Resumed from {ckpts[-1]}  (completed epoch {ckpt['epoch']})")
    else:
        print("No checkpoint found — starting from scratch.")

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, N_EPOCHS + 1):

    # Train
    model.train()
    total_train = 0.0
    for batch_idx, (x, y) in enumerate(train_loader, 1):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_train += loss.item()

        if batch_idx % LOG_EVERY == 0 or batch_idx == len(train_loader):
            avg_so_far = total_train / batch_idx
            msg = (
                f"  Epoch {epoch:>2}/{N_EPOCHS}  "
                f"batch {batch_idx:>4}/{len(train_loader)}  "
                f"train loss: {avg_so_far:.4f}"
            )
            print(f"{msg:<70}", end='\r', flush=True)

    # Validate
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total_val += criterion(logits.view(-1, VOCAB_SIZE), y.view(-1)).item()

    train_loss = total_train / len(train_loader)
    val_loss   = total_val   / len(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Print epoch summary on a fresh line (clears the batch progress line)
    print(f"\rEpoch {epoch:>2}/{N_EPOCHS}  |  train: {train_loss:.4f}  |  val: {val_loss:.4f}{'':30}")

    # Save checkpoint after every epoch
    torch.save({
        "epoch"               : epoch,
        "model_state_dict"    : model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses"        : train_losses,
        "val_losses"          : val_losses,
    }, f"model_{ARCH}_ckpt_{epoch:02d}.pt")

# Save final weights for the viz notebook
torch.save(model.state_dict(), f"model_{ARCH}.pt")
print(f"\nSaved model_{ARCH}.pt")

## Loss Curves

In [ ]:
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="Train")
plt.plot(epochs, val_losses,   label="Val")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title(f"{ARCH.upper()} — training curves")
plt.legend()
plt.tight_layout()
plt.show()

## Generate Text

Sample from the trained model given a seed string.  
Lower `temperature` → more repetitive but more "on theme". Higher → more creative but noisier.

In [ ]:
@torch.no_grad()
def generate(seed: str, length: int = 300, temperature: float = 1.0) -> str:
    model.eval()
    tokens = [char_to_idx[ch] for ch in seed if ch in char_to_idx]
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)

    output = seed
    for _ in range(length):
        inp    = tokens[:, -CONTEXT_LEN:]          # never exceed context window
        logits = model(inp)                         # (1, T, vocab_size)
        probs  = torch.softmax(logits[0, -1] / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        output += idx_to_char[next_token]
        tokens = torch.cat(
            [tokens, torch.tensor([[next_token]], device=DEVICE)], dim=1
        )
    return output


seed        = "The fear of the Lord is"
temperature = 0.8

print(generate(seed, length=300, temperature=temperature))